In [1]:
import numpy as np
import cv2

In [2]:
face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

In [3]:
# Read image with alpha channel (transparency)
# cv2.IMREAD_UNCHANGED - Keeps 4 channels (BGR + Alpha)
sunglasses = cv2.imread("sunglasses.webp", cv2.IMREAD_UNCHANGED)

In [4]:
sunglasses.shape

(350, 931, 4)

In [5]:
cap = cv2.VideoCapture(0)
while True:
    flag, frame = cap.read()
    if not flag:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3)
    for x,y,w,h in faces:
        # Resize sunglasses according to face width
        # width = face width
        # height = 40% of face height
        overlay_width = w
        overlay_height = int(h * 0.4)
        resized_sunglasses = cv2.resize(sunglasses, 
                                        (overlay_width, overlay_height))
        x_offset = x
        y_offset = y + h // 4

        # Handle Transparency (Alpha Blending)
        # first 3 channels - color
        overlay_img = resized_sunglasses[:, :, :3]
        
        # Last channel - alpha
        mask = resized_sunglasses[:, :, 3]

        # Create inverse mask (for background)
        mask_inv = cv2.bitwise_not(mask)

        # Define ROI - Region Of Interest
        roi = frame[y_offset:y_offset + overlay_height, 
            x_offset:x_offset + overlay_width]

        bg = cv2.bitwise_and(roi, roi, mask=mask_inv)
        fg = cv2.bitwise_and(overlay_img, overlay_img, mask=mask)

        combined = cv2.add(bg, fg)
        frame[y_offset:y_offset + overlay_height, 
            x_offset:x_offset + overlay_width] = combined
        
        cv2.rectangle(frame, (x,y), (x+w, y+h), (0,255,0), 2)
    cv2.imshow("sunglasses filter", frame)
    key = cv2.waitKey(1) & 0xff
    if key == 27:
        break
cap.release()
cv2.destroyAllWindows()